# Seed Variance: how much do two identical runs differ?

Re-trains `pcrtc/09` unchanged except for the weight-initialisation seed,
on the **identical** spatial-block split, to measure run-to-run variability
directly.

**Why this matters more than another architecture.** The DEM experiment
accidentally produced one estimate: two near-identical models, differing
only in initialisation, landed **0.05 ZNCC apart** on the same 255
validation patches. That single pair is an indication of scale, not a
variance. But even as an indication it is decisive for how every other
result in this study should be read -- the headline in-region ZNCC is
0.2344, so a 0.05 spread is a ±20% band, and no comparison narrower than
that is currently credible.

This notebook turns n = 1 into n = 2 pairs at the cost of one training run
and no new code paths. That strengthens **every** comparison in the
dissertation, which no single new architecture could do.

**What is held fixed, and what is not.**

| | value | why |
|---|---|---|
| `SPLIT_SEED` | 42, fixed | the split must be *identical* to `09`'s, or the comparison is meaningless |
| `INIT_SEED` | varied | weight init, batch order, timestep draws -- the thing being measured |
| everything else | identical to `09` | architecture, data, epochs, LR, schedule, sampler |

The split uses its own `random.Random(SPLIT_SEED)` instance, so it is
unaffected by the global seed that `seed_everything(INIT_SEED)` sets. The
notebook asserts 534/255 before training starts; if that assertion fails,
the split drifted and nothing below is comparable.

**Cost:** one training run, same duration as `09`. Runs unattended.

**This notebook does not execute automatically. Run cells top to bottom.**

## GPU configuration

In [1]:
import os
import sys
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4090


## Configuration

Every value below matches `pcrtc/09` except `INIT_SEED`. Change only
`INIT_SEED` if you want to run a third replicate later.

In [2]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'
REGION = 'tuk'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 42     # MUST stay 42 -- identical split to 09
INIT_SEED = 43      # the only thing that varies

CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
LIDAR_SURVEY_DATE = __import__('datetime').date(2024, 4, 16)
BLOCK_SIZE_M = 1024.0
BUFFER_M = 150.0

CHECKPOINT_NAME = f's1_{REGION}_pcrtc_realattrs_spatialsplit_seed{INIT_SEED}_unet_best.pth'
METRICS_FILENAME = f's1_pcrtc_realattrs_spatialsplit_seed{INIT_SEED}_validation_metrics.json'
BASELINE_METRICS = 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'      # 09
DEM_METRICS = 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json'       # 03 (inert DEM)

print(f'Replicate of pcrtc/09 with INIT_SEED={INIT_SEED}, SPLIT_SEED={SPLIT_SEED}')
print('Checkpoint:', CHECKPOINT_DIR / CHECKPOINT_NAME)

Replicate of pcrtc/09 with INIT_SEED=43, SPLIT_SEED=42
Checkpoint: /cs/student/project_msc/2025/aibh/jiayiche/checkpoints/s1_tuk_pcrtc_realattrs_spatialsplit_seed43_unet_best.pth


## Imports and seeding

In [3]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(INIT_SEED)   # NOT SPLIT_SEED -- this is the variable under test
print('Global seed set to', INIT_SEED)

Global seed set to 43


## Dataset adapter -- unchanged from `09`

In [4]:
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_norm = (acq_date - LIDAR_SURVEY_DATE).days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256)):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        with rasterio.open(self.lidar_dir / f'lidar_patch_{patch_id}.tif') as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask),
                's1': condition.float(), 'attrs': attrs,
                'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Spatial-block split -- `SPLIT_SEED`, not `INIT_SEED`

Copied verbatim from `09`. The only edit is `random.Random(SPLIT_SEED)`
in place of `random.Random(SEED)`, which keeps the split pinned to 42
while the initialisation seed varies. The assertion below is the guard
that makes this whole experiment valid.

In [5]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'

def patch_centroid(patch_id):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{patch_id}.tif') as src:
        b = src.bounds
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks = {}
dropped_buffer = []
for pid, (cx, cy) in centroids.items():
    bid, boundary_dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if boundary_dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

block_ids = list(blocks.keys())
random.Random(SPLIT_SEED).shuffle(block_ids)   # pinned to 42, independent of INIT_SEED

target_val_patches = int(len(paired_ids) * VAL_FRACTION)
val_ids, train_ids = [], []
running_val_count = 0
for bid in block_ids:
    if running_val_count < target_val_patches:
        val_ids.extend(blocks[bid])
        running_val_count += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'Split: train={len(train_ids)}, val={len(val_ids)}, dropped={len(dropped_buffer)}')
assert (len(train_ids), len(val_ids)) == (534, 255), (
    f'Split is {len(train_ids)}/{len(val_ids)}, expected 534/255. The split has drifted '
    f'from 09 and no comparison below would be valid. Stop and investigate.')
print('Split confirmed identical to 09.')

train_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, train_ids, CONTEXT_K, TARGET_HW)
val_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

Split: train=534, val=255, dropped=887
Split confirmed identical to 09.


## Model, scheduler, optimizer

In [6]:
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K,
                        base_channels=128, embed_dim=256, unet_depth=4,
                        attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Trainable parameters: 104,625,307


## Training loop -- identical to `09`

`09`'s best validation loss arrived at **epoch 93 of 100**, with validation
wandering in the 0.0137-0.0174 band throughout and rising streaks of up to
3 epochs. Early validation movement carries no signal here. Do not stop
this run early.

In [7]:
from torch.cuda.amp import autocast, GradScaler

def masked_mse(prediction, target, mask):
    valid = mask.bool().unsqueeze(1)
    error = (prediction - target) ** 2
    return error[valid].mean()

scaler = GradScaler()
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')
best_epoch = -1
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    for batch in train_loader:
        target = batch['lidar'].to(DEVICE, non_blocking=True)
        condition = batch['s1'].to(DEVICE, non_blocking=True)
        attrs = batch['attrs'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            noisy = scheduler.q_sample(target, timestep)
            prediction = model(noisy, condition, attrs, timestep)
            loss = masked_mse(prediction, target, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_total += loss.item()
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['lidar'].to(DEVICE, non_blocking=True)
            condition = batch['s1'].to(DEVICE, non_blocking=True)
            attrs = batch['attrs'].to(DEVICE, non_blocking=True)
            mask = batch['mask'].to(DEVICE, non_blocking=True)
            timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            with autocast():
                prediction = model(scheduler.q_sample(target, timestep), condition, attrs, timestep)
                val_total += masked_mse(prediction, target, mask).item()
    train_loss = train_total / max(1, len(train_loader))
    val_loss = val_total / max(1, len(val_loader))
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    print(f'Epoch {epoch + 1:03d}/{EPOCHS}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch + 1
        torch.save({'model_state_dict': model.state_dict(),
                    'config': {'context_k': CONTEXT_K, 'timesteps': TIMESTEPS,
                               'noise_schedule': NOISE_SCHEDULE, 'region': REGION,
                               'split_seed': SPLIT_SEED, 'init_seed': INIT_SEED},
                    'epoch': best_epoch, 'val_loss': val_loss},
                   CHECKPOINT_DIR / CHECKPOINT_NAME)

print(f'\nBest val {best_val:.6f} at epoch {best_epoch}  (09 reached 0.013706 at epoch 93)')

/tmp/ipykernel_198084/1107549189.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_198084/1107549189.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_198084/1107549189.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/100: train=0.096641 val=0.028169
Epoch 002/100: train=0.021924 val=0.022187
Epoch 003/100: train=0.018085 val=0.018952
Epoch 004/100: train=0.015461 val=0.017605
Epoch 005/100: train=0.014737 val=0.015976
Epoch 006/100: train=0.014332 val=0.017324
Epoch 007/100: train=0.014987 val=0.016000
Epoch 008/100: train=0.013892 val=0.015983
Epoch 009/100: train=0.013699 val=0.015690
Epoch 010/100: train=0.013354 val=0.016943
Epoch 011/100: train=0.014095 val=0.015830
Epoch 012/100: train=0.014653 val=0.015984
Epoch 013/100: train=0.013973 val=0.014788
Epoch 014/100: train=0.013646 val=0.016793
Epoch 015/100: train=0.013562 val=0.016939
Epoch 016/100: train=0.013598 val=0.015630
Epoch 017/100: train=0.013282 val=0.016843
Epoch 018/100: train=0.013567 val=0.013493
Epoch 019/100: train=0.013481 val=0.015270
Epoch 020/100: train=0.013717 val=0.015412
Epoch 021/100: train=0.013268 val=0.016309
Epoch 022/100: train=0.013109 val=0.014697
Epoch 023/100: train=0.013513 val=0.016540
Epoch 024/1

## Evaluation -- identical metric suite and sampler

In [8]:
checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
metric_rows = []
with torch.no_grad():
    for batch in val_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            metric_rows.append({
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            })

metrics_path = OUTPUT_DIR / METRICS_FILENAME
with metrics_path.open('w') as h:
    json.dump(metric_rows, h, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {k: float(np.nanmean([r[k] for r in metric_rows])) for k in metric_rows[0] if k != 'patch_id'})

Saved: /cs/student/project_msc/2025/aibh/jiayiche/s1_training_outputs/s1_pcrtc_realattrs_spatialsplit_seed43_validation_metrics.json
Mean metrics: {'rmse_m': 0.20844222096835865, 'bias_m': 0.024682806917082738, 'sigma_error_pct': 21.03748498059371, 'normal_angle_error_deg': 2.078961368635589, 'jsd': 0.08909023201056555, 'psd_rmse': 1.2661808948890836, 'zncc': 0.2644985278217777, 'gt_std_val': 0.17209622516643766, 'pred_std_val': 0.16988310057158565}


## The variance estimate

Three runs, all on the identical split: `09`, the inert-DEM run (which is
effectively a second seed, since its conditioning branch carried no
information), and this one. Three pairwise differences.

**Report the spread, not a p-value.** With three runs you have an honest
range, not a distribution. The number to carry into the write-up is the
largest pairwise |ΔZNCC| -- the threshold below which a difference between
configurations should not be claimed.

In [9]:
def load_rows(name):
    p = OUTPUT_DIR / name
    if not p.exists():
        print(f'  (missing: {name})')
        return None
    return {r['patch_id']: r for r in json.load(p.open())}

runs = {'09 (seed 42)': load_rows(BASELINE_METRICS),
        'inert-DEM run': load_rows(DEM_METRICS),
        f'replicate (seed {INIT_SEED})': {r['patch_id']: r for r in metric_rows}}
runs = {k: v for k, v in runs.items() if v}

common = set.intersection(*(set(v) for v in runs.values()))
print(f'runs: {len(runs)}   patches common to all: {len(common)}\n')

METRICS = ['zncc', 'rmse_m', 'psd_rmse', 'jsd', 'pred_std_val']
print(f"{'metric':<16}" + ''.join(f'{k:>28}' for k in runs))
for m in METRICS:
    line = f'{m:<16}'
    for k, rows in runs.items():
        line += f'{np.nanmean([rows[i][m] for i in common]):>28.4f}'
    print(line)

print('\n--- pairwise differences (the variance estimate) ---')
names = list(runs)
zncc_spread = []
for a in range(len(names)):
    for b in range(a + 1, len(names)):
        na, nb = names[a], names[b]
        d = np.array([runs[nb][i]['zncc'] - runs[na][i]['zncc'] for i in common])
        d = d[np.isfinite(d)]
        zncc_spread.append(abs(float(d.mean())))
        print(f'  |zncc| {na} vs {nb}: {abs(d.mean()):.4f}')

worst = max(zncc_spread)
print(f'\nLargest pairwise |delta ZNCC| across runs: {worst:.4f}')
print(f'In-region headline ZNCC is 0.2344, so this is a {worst / 0.2344:.0%} band.')
print('\nUse this as the threshold in the write-up: differences between')
print('configurations smaller than this are not claimed as effects.')

runs: 3   patches common to all: 255

metric                          09 (seed 42)               inert-DEM run         replicate (seed 43)
zncc                                  0.2344                      0.2866                      0.2645
rmse_m                                0.1945                      0.1894                      0.2084
psd_rmse                              1.3092                      1.6886                      1.2662
jsd                                   0.1180                      0.1334                      0.0891
pred_std_val                          0.1384                      0.1418                      0.1699

--- pairwise differences (the variance estimate) ---
  |zncc| 09 (seed 42) vs inert-DEM run: 0.0522
  |zncc| 09 (seed 42) vs replicate (seed 43): 0.0301
  |zncc| inert-DEM run vs replicate (seed 43): 0.0221

Largest pairwise |delta ZNCC| across runs: 0.0522
In-region headline ZNCC is 0.2344, so this is a 22% band.

Use this as the threshold in the write